# 06 - Decoding sweep: can the repetition loops be fixed without retraining?

> Run with **Save Version -> Save & Run All (Commit)**. Settings: **Accelerator = GPU T4 x2**, **Internet = On**.
> **Input (Add Input):** only the adapters dataset `nlp-qna-adapters` (the `nn/` adapter). The 50 test questions are embedded below.

**Why.** In Phase 5, 130 of 300 hold-out answers hit the 768-token cap, and most of those were repetition loops that start early
(median at word ~130), so the cap is not the cause. Greedy decoding with a near-zero repetition penalty is a known cause of loops.

**What.** The same trained adapter (`nn`) and the **exact Phase 5 prompts** for the first 50 of the 300 Phase 5 hold-out questions are answered
under 7 decoding settings (all with a 1,280-token cap):

| config | decoding |
|---|---|
| A | greedy, HF repetition penalty 1.05 (= Phase 5 decoding, only the cap is higher) |
| B | greedy, HF repetition penalty 1.15 (HF's penalty also hits words in the prompt) |
| C | greedy, penalty 1.15 on the answer's own tokens only |
| D | greedy, penalty 1.05 + no repeated 12-token n-gram inside the answer |
| E | greedy, answer-only penalty 1.1 + no repeated 12-token n-gram |
| F | sampling (temperature 0.7, top-p 0.9), penalty 1.05 |
| G | sampling + answer-only penalty 1.1 + no repeated 12-token n-gram |

Scored with the same harness as Phase 5 against the human reference answers, next to B0a (copy the nearest answer) and the saved
Phase 5 answers for the same 50 questions. **N = 50, so differences under ~0.02 ROUGE-L are noise; the table chooses a setting, it is not a final result.**

In [ ]:
!pip install -q -U peft bitsandbytes accelerate huggingface_hub rouge-score sacrebleu

In [ ]:
%%writefile infer.py
"""Phase 5 generation worker: one process per GPU.

Reads a jobs file (jsonl: uid, adapter, prio, system, user), generates greedy answers in batches, and appends every
finished batch to --out immediately, so a crashed or time-limited run resumes where it stopped (done uids are skipped).
adapter is "nn" / "none" (a LoRA folder under --adapter_dir) or "base" (the same model with the adapter switched off:
the zero-shot baseline B1).  Pure ASCII source on purpose.

    CUDA_VISIBLE_DEVICES=0 python infer.py --jobs jobs.jsonl --adapter_dir /path --adapters nn --out gen_gpu0.jsonl
"""
import argparse
import json
import os
import time

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, LogitsProcessor, LogitsProcessorList
from peft import PeftModel


def parse():
    p = argparse.ArgumentParser()
    p.add_argument("--jobs", required=True)
    p.add_argument("--adapter_dir", required=True)
    p.add_argument("--adapters", required=True, help="comma list handled by this worker, e.g. none,base")
    p.add_argument("--out", required=True)
    p.add_argument("--model", default="Qwen/Qwen2.5-3B-Instruct")
    p.add_argument("--max_new", type=int, default=768)
    p.add_argument("--batch", type=int, default=12)
    p.add_argument("--rep_penalty", type=float, default=1.05, help="HF repetition penalty (acts on prompt AND answer tokens)")
    p.add_argument("--gen_rep_penalty", type=float, default=1.0, help="penalty on tokens of the ANSWER only (1.0 = off)")
    p.add_argument("--gen_no_repeat", type=int, default=0, help="forbid repeating any n-gram of the ANSWER (0 = off)")
    p.add_argument("--do_sample", action="store_true")
    p.add_argument("--temperature", type=float, default=0.7)
    p.add_argument("--top_p", type=float, default=0.9)
    p.add_argument("--seed", type=int, default=1234)
    p.add_argument("--hours", type=float, default=10.0)
    p.add_argument("--no_quant", action="store_true")
    return p.parse_args()


class GenRepetitionPenalty(LogitsProcessor):
    """Repetition penalty over the generated tokens only (the HF one also penalises every prompt token, which
    discourages copying wording from the reference answers in the prompt). Assumes left padding."""

    def __init__(self, penalty, prompt_len):
        self.p, self.plen = penalty, prompt_len

    def __call__(self, input_ids, scores):
        g = input_ids[:, self.plen:]
        if g.shape[1] == 0:
            return scores
        s = torch.gather(scores, 1, g)
        s = torch.where(s < 0, s * self.p, s / self.p)
        return scores.scatter(1, g, s)


class GenNoRepeatNGram(LogitsProcessor):
    """Forbid any n-gram that already occurs in the generated answer (prompt excluded). Incremental per row."""

    def __init__(self, n, prompt_len):
        self.n, self.plen, self.seen = n, prompt_len, None

    def __call__(self, input_ids, scores):
        B, L = input_ids.shape
        if self.seen is None:
            self.seen = [dict() for _ in range(B)]
        glen = L - self.plen
        if glen < self.n - 1:
            return scores
        rows = input_ids[:, self.plen:].tolist()
        for b in range(B):
            g = rows[b]
            if glen >= self.n:                      # register the newest complete n-gram
                self.seen[b].setdefault(tuple(g[-self.n:-1]), set()).add(g[-1])
            banned = self.seen[b].get(tuple(g[len(g) - self.n + 1:]))
            if banned:
                scores[b, list(banned)] = -float("inf")
        return scores


def load_model(a, names):
    cuda = torch.cuda.is_available()
    if a.no_quant:
        base = AutoModelForCausalLM.from_pretrained(a.model, torch_dtype=torch.float16 if cuda else torch.float32)
        if cuda:
            base = base.to("cuda:0")
    else:
        bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16,
                                 bnb_4bit_use_double_quant=True)
        base = AutoModelForCausalLM.from_pretrained(a.model, quantization_config=bnb, torch_dtype=torch.float16,
                                                    device_map={"": 0})
    real = [n for n in names if n != "base"]
    if not real:
        return base.eval(), False
    model = PeftModel.from_pretrained(base, os.path.join(a.adapter_dir, real[0]), adapter_name=real[0])
    for n in real[1:]:
        model.load_adapter(os.path.join(a.adapter_dir, n), adapter_name=n)
    return model.eval(), True


def generate(model, tok, texts, a, device):
    enc = tok(texts, return_tensors="pt", padding=True, add_special_tokens=False).to(device)
    plen = enc["input_ids"].shape[1]
    procs = LogitsProcessorList()
    if getattr(a, "gen_rep_penalty", 1.0) != 1.0:
        procs.append(GenRepetitionPenalty(a.gen_rep_penalty, plen))
    if getattr(a, "gen_no_repeat", 0):
        procs.append(GenNoRepeatNGram(a.gen_no_repeat, plen))
    kw = dict(max_new_tokens=a.max_new, repetition_penalty=a.rep_penalty, pad_token_id=tok.pad_token_id,
              logits_processor=procs)
    if getattr(a, "do_sample", False):
        torch.manual_seed(a.seed)
        kw.update(do_sample=True, temperature=a.temperature, top_p=a.top_p, top_k=0)
    else:
        kw.update(do_sample=False, temperature=None, top_p=None, top_k=None)
    with torch.no_grad():
        out = model.generate(**enc, **kw)
    gen = out[:, enc["input_ids"].shape[1]:]
    res = []
    for g in gen:
        ids = g.tolist()
        n = len(ids)
        while n and ids[n - 1] == tok.pad_token_id:
            n -= 1
        res.append((tok.decode(ids[:n], skip_special_tokens=True).strip(), n))
    return res


def main():
    a = parse()
    names = a.adapters.split(",")
    jobs = [json.loads(l) for l in open(a.jobs, encoding="utf-8")]
    jobs = [j for j in jobs if j["adapter"] in names]
    done = set()
    if os.path.exists(a.out):
        done = {json.loads(l)["uid"] for l in open(a.out, encoding="utf-8") if l.strip()}
    todo = [j for j in jobs if j["uid"] not in done]
    print("jobs for %s: %d | already done: %d | to do: %d" % (names, len(jobs), len(done), len(todo)), flush=True)
    if not todo:
        return
    tok = AutoTokenizer.from_pretrained(a.model)
    tok.padding_side = "left"
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    for j in todo:
        j["text"] = tok.apply_chat_template([{"role": "system", "content": j["system"]}, {"role": "user", "content": j["user"]}],
                                            tokenize=False, add_generation_prompt=True)
        j["plen"] = len(tok(j["text"], add_special_tokens=False)["input_ids"])
    device = "cuda:0" if torch.cuda.is_available() else "cpu"
    model, has_peft = load_model(a, names)
    t_end = time.time() + a.hours * 3600
    t0 = time.time()
    n_done = 0
    fout = open(a.out, "a", encoding="utf-8")
    for name in names:
        group = sorted((j for j in todo if j["adapter"] == name), key=lambda j: (j["prio"], -j["plen"]))
        if not group:
            continue
        if name != "base" and has_peft:
            model.set_adapter(name)
        print("adapter %s: %d jobs" % (name, len(group)), flush=True)
        i = 0
        bs = a.batch
        while i < len(group):
            if time.time() > t_end:
                print("time limit reached - stopping; rerun to resume", flush=True)
                return
            batch = group[i:i + bs]
            try:
                if name == "base" and has_peft:
                    with model.disable_adapter():
                        res = generate(model, tok, [j["text"] for j in batch], a, device)
                else:
                    res = generate(model, tok, [j["text"] for j in batch], a, device)
            except torch.cuda.OutOfMemoryError:
                torch.cuda.empty_cache()
                if bs == 1:
                    raise
                bs = max(1, bs // 2)
                print("out of memory - batch size now", bs, flush=True)
                continue
            for j, (text, n) in zip(batch, res):
                fout.write(json.dumps({"uid": j["uid"], "adapter": name, "text": text, "n_tokens": n,
                                       "hit_limit": n >= a.max_new}, ensure_ascii=False) + "\n")
            fout.flush()
            i += len(batch)
            n_done += len(batch)
            el = time.time() - t0
            print("[%5.0f min] %s %d/%d | %.1f s per answer" % (el / 60, name, i, len(group), el / n_done), flush=True)
    print("DONE", n_done, "answers in %.1f min" % ((time.time() - t0) / 60), flush=True)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile decode_sweep.py
"""Decoding sweep worker: one process per GPU. Loads the base model + one adapter once, then answers the same prompts
under several decoding configurations. Appends every batch to --out (resumable: done uids are skipped).

    CUDA_VISIBLE_DEVICES=0 python decode_sweep.py --payload payload.json --configs A,C --adapter_dir /path --out g0.jsonl
Pure ASCII source on purpose.
"""
import argparse
import json
import os
import time
from types import SimpleNamespace

import torch
from transformers import AutoTokenizer

import infer

# name: decoding settings. max_new 1280 for all (the 768 cap cut ~40% of references short).
BASE = dict(max_new=1280, rep_penalty=1.0, gen_rep_penalty=1.0, gen_no_repeat=0, do_sample=False, temperature=0.7, top_p=0.9)
CONFIGS = {
    "A_greedy_rp1.05": dict(rep_penalty=1.05),                                    # Phase 5 decoding, higher cap only
    "B_greedy_rp1.15_hf": dict(rep_penalty=1.15),                                  # standard HF penalty (also hits prompt words)
    "C_greedy_genrp1.15": dict(gen_rep_penalty=1.15),                              # penalty on the answer's own tokens only
    "D_greedy_rp1.05_nr12": dict(rep_penalty=1.05, gen_no_repeat=12),              # no repeated 12-token n-gram in the answer
    "E_greedy_genrp1.1_nr12": dict(gen_rep_penalty=1.1, gen_no_repeat=12),
    "F_sample_t0.7_rp1.05": dict(rep_penalty=1.05, do_sample=True),
    "G_sample_t0.7_genrp1.1_nr12": dict(gen_rep_penalty=1.1, gen_no_repeat=12, do_sample=True),
}


def settings(name, max_new=None):
    s = dict(BASE, **CONFIGS[name])
    if max_new:
        s["max_new"] = max_new
    return s


def main():
    p = argparse.ArgumentParser()
    p.add_argument("--payload", required=True)
    p.add_argument("--configs", required=True)
    p.add_argument("--adapter_dir", required=True)
    p.add_argument("--adapter", default="nn")
    p.add_argument("--out", required=True)
    p.add_argument("--model", default="Qwen/Qwen2.5-3B-Instruct")
    p.add_argument("--batch", type=int, default=10)
    p.add_argument("--max_new", type=int, default=0, help="override max_new (smoke tests)")
    p.add_argument("--no_quant", action="store_true")
    a = p.parse_args()

    items = json.load(open(a.payload, encoding="utf-8"))["items"]
    names = a.configs.split(",")
    done = set()
    if os.path.exists(a.out):
        done = {json.loads(l)["uid"] for l in open(a.out, encoding="utf-8") if l.strip()}
    tok = AutoTokenizer.from_pretrained(a.model)
    tok.padding_side = "left"
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    for it in items:
        it["text"] = tok.apply_chat_template([{"role": "system", "content": it["system"]}, {"role": "user", "content": it["user"]}],
                                             tokenize=False, add_generation_prompt=True)
        it["plen"] = len(tok(it["text"], add_special_tokens=False)["input_ids"])
    items.sort(key=lambda it: -it["plen"])
    device = "cuda:0" if torch.cuda.is_available() else "cpu"
    model, _ = infer.load_model(SimpleNamespace(no_quant=a.no_quant, model=a.model, adapter_dir=a.adapter_dir), [a.adapter])
    model.set_adapter(a.adapter)
    fout = open(a.out, "a", encoding="utf-8")
    for name in names:
        s = settings(name, a.max_new or None)
        todo = [it for it in items if "%s|%s" % (name, it["qid"]) not in done]
        print("config %s: %s | to do %d" % (name, s, len(todo)), flush=True)
        t0, bs, i, bi = time.time(), a.batch, 0, 0
        while i < len(todo):
            batch = todo[i:i + bs]
            ga = SimpleNamespace(seed=1234 + bi, **s)
            tb = time.time()
            try:
                res = infer.generate(model, tok, [it["text"] for it in batch], ga, device)
            except torch.cuda.OutOfMemoryError:
                torch.cuda.empty_cache()
                if bs == 1:
                    raise
                bs = max(1, bs // 2)
                print("out of memory - batch size now", bs, flush=True)
                continue
            sec = (time.time() - tb) / len(batch)
            for it, (text, n) in zip(batch, res):
                fout.write(json.dumps({"uid": "%s|%s" % (name, it["qid"]), "config": name, "qid": it["qid"], "text": text,
                                       "n_tokens": n, "hit_limit": n >= s["max_new"], "sec_per_answer": sec}, ensure_ascii=False) + "\n")
            fout.flush()
            i += len(batch)
            bi += 1
            print("[%5.1f min] %s %d/%d | %.1f s per answer" % ((time.time() - t0) / 60, name, i, len(todo), sec), flush=True)
    print("DONE", flush=True)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile eval_harness.py
"""Single evaluation harness used by every system (baselines, ablations, final model).

    from eval_harness import evaluate
    scores = evaluate(preds, refs)            # dicts {question_id: text}
    scores = evaluate(preds, refs, bertscore=True, meteor=True)   # slower extras

Metrics: ROUGE-1/2/L F1, corpus BLEU (sacrebleu), length ratio, agronomic fact recall/precision/F1
(number+unit pairs, formulation-tagged chemicals such as "dinotefuran 20 SG", binomials), and optionally
METEOR (nltk) and BERTScore-F1. Extras are lazy-imported so the core runs with rouge-score + sacrebleu only.
"""
import re
import statistics as st

from rouge_score import rouge_scorer

_UNIT = {"gm": "g", "gram": "g", "grams": "g", "gms": "g", "litre": "l", "litres": "l", "lit": "l", "liter": "l",
         "liters": "l", "hectare": "ha", "hectares": "ha", "per cent": "%", "percent": "%", "quintals": "q",
         "quintal": "q", "tonnes": "t", "tonne": "t", "day": "days", "kgs": "kg"}
NUM_UNIT_RE = re.compile(
    r"(\d+(?:\.\d+)?)\s*(?:-|\u2013|to)?\s*(\d+(?:\.\d+)?)?\s*"
    r"(kg|kgs|gms?|grams?|g|mg|ml|litres?|liters?|lit|l|quintals?|q|tonnes?|t|ha|hectares?|acre|cm|mm|"
    r"days?|per\s*cent|percent|%|ppm)\b"
    r"(?:\s*(?:/|per)\s*(ha|hectare|acre|litres?|liters?|lit|l|plant|kg|m2|10\s*l))?", re.I)
CHEM_RE = re.compile(r"\b([A-Za-z][A-Za-z\-]{3,})\s+\d+(?:\.\d+)?\s*%?\s*"
                     r"(SG|SL|EC|WP|WG|SC|FS|DP|GR|ZC|OD|EW|CS|ME|WS|DF)\b")
BINOM_RE = re.compile(r"\(([A-Z][a-z]{3,}\s+[a-z]{4,})\)")


def _u(x):
    x = re.sub(r"\s+", " ", x.lower().strip())
    return _UNIT.get(x, x)


def extract_facts(text):
    facts = set()
    for m in NUM_UNIT_RE.finditer(text):
        a, b, unit, per = m.groups()
        facts.add("%s%s%s%s" % (a, "-" + b if b else "", _u(unit), "/" + _u(per) if per else ""))
    for m in CHEM_RE.finditer(text):
        facts.add("chem:" + m.group(1).lower())
    for m in BINOM_RE.finditer(text):
        facts.add("sp:" + m.group(1).lower())
    return facts


def _fact_scores(preds, refs):
    r_list, p_list, f_list = [], [], []
    for p, r in zip(preds, refs):
        fr, fp = extract_facts(r), extract_facts(p)
        if not fr:
            continue
        inter = len(fr & fp)
        rec = inter / len(fr)
        prec = inter / len(fp) if fp else 0.0
        r_list.append(rec)
        p_list.append(prec)
        f_list.append(0.0 if rec + prec == 0 else 2 * rec * prec / (rec + prec))
    n = max(1, len(r_list))
    return {"fact_recall": sum(r_list) / n, "fact_precision": sum(p_list) / n, "fact_f1": sum(f_list) / n,
            "fact_n_refs": len(r_list)}


def evaluate(preds, refs, bertscore=False, meteor=False, bert_model="roberta-large"):
    ids = [i for i in refs if i in preds]
    P = [preds[i] or "" for i in ids]
    R = [refs[i] for i in ids]
    sc = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
    agg = {"rouge1": 0.0, "rouge2": 0.0, "rougeL": 0.0}
    for p, r in zip(P, R):
        s = sc.score(r, p)
        for k in agg:
            agg[k] += s[k].fmeasure
    n = max(1, len(ids))
    out = {"n": len(ids), "n_missing": len(refs) - len(ids)}
    out.update({k: v / n for k, v in agg.items()})
    import sacrebleu
    out["bleu"] = sacrebleu.corpus_bleu(P, [R]).score / 100.0
    pl, rl = [len(p.split()) for p in P], [len(r.split()) for r in R]
    out["len_ratio_mean"] = (sum(pl) / n) / max(1e-9, sum(rl) / n)
    out["len_ratio_median_per_item"] = st.median([a / max(1, b) for a, b in zip(pl, rl)]) if ids else 0.0
    out.update(_fact_scores(P, R))
    if meteor:
        from nltk.translate.meteor_score import meteor_score
        out["meteor"] = sum(meteor_score([r.split()], p.split()) for p, r in zip(P, R)) / n
    if bertscore:
        from bert_score import score as bs
        _, _, f1 = bs(P, R, model_type=bert_model, lang="en", batch_size=16, verbose=False)
        out["bertscore_f1"] = float(f1.mean())
    return out


def evaluate_by_group(preds, refs, groups, **kw):
    """groups: {name: set(question_ids)} -> {name: scores}; e.g. novel vs repeated, or per crop."""
    return {g: evaluate(preds, {i: refs[i] for i in ids if i in refs}, **kw) for g, ids in groups.items()}


if __name__ == "__main__":
    t = "Spray dinotefuran 20 SG @ 60 g/acre; seed rate 25 kg/ha. Pest: Amrasca devastans (Amrasca devastans)."
    print(extract_facts(t))
    print(evaluate({"a": t}, {"a": t}))

## 0. The 50 test questions (embedded: prompts, references, B0a and Phase 5 answers)

In [ ]:
PAYLOAD_B64 = """
...
"""
import base64, gzip, json
payload = json.loads(gzip.decompress(base64.b64decode("".join(PAYLOAD_B64.split()))).decode("utf-8"))
json.dump(payload, open("payload.json", "w", encoding="utf-8"), ensure_ascii=False)
print(payload["source"], "| items:", len(payload["items"]))

## 1. Configuration

In [ ]:
import glob, os, sys, json, time, subprocess, collections as C
_missing = [f for f in ("infer.py", "decode_sweep.py", "eval_harness.py", "payload.json") if not os.path.exists(f)]
assert not _missing, "missing " + str(_missing) + " - use Run All from the top"
sys.path.insert(0, os.getcwd())
import torch
assert torch.cuda.is_available(), "No GPU. Right panel -> Session options -> Accelerator -> GPU T4 x2, then Run All."
print("GPUs:", torch.cuda.device_count(), [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

MODEL = "Qwen/Qwen2.5-3B-Instruct"
BATCH = 10
WORK = "/kaggle/working"
hit = glob.glob("/kaggle/input/**/nn/adapter_config.json", recursive=True)
assert hit, "attach the adapters dataset (nlp-qna-adapters) with the nn/ folder"
AD_DIR = os.path.dirname(os.path.dirname(hit[0]))
import decode_sweep as DS
ALL = list(DS.CONFIGS)
if torch.cuda.device_count() >= 2:
    workers = [[c for c in ALL if c[0] in "ACEG"], [c for c in ALL if c[0] in "BDF"]]
else:
    workers = [ALL]
print("adapters:", AD_DIR)
for i, w in enumerate(workers):
    print("GPU", i, "->", w)

## 2. Generate (one worker per GPU, resumable)

In [ ]:
from huggingface_hub import snapshot_download
snapshot_download(MODEL)
os.makedirs(WORK + "/sweep", exist_ok=True)
procs = []
for gpu, names in enumerate(workers):
    log = open("%s/sweep/worker%d.log" % (WORK, gpu), "a")
    cmd = [sys.executable, "decode_sweep.py", "--payload", "payload.json", "--configs", ",".join(names), "--adapter_dir", AD_DIR,
           "--out", "%s/sweep/gen_gpu%d.jsonl" % (WORK, gpu), "--model", MODEL, "--batch", str(BATCH)]
    procs.append((gpu, subprocess.Popen(cmd, env=dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu), TOKENIZERS_PARALLELISM="false"),
                                        stdout=log, stderr=subprocess.STDOUT)))
t0 = time.time()
while any(p.poll() is None for _, p in procs):
    time.sleep(90)
    for gpu, p in procs:
        lines = open("%s/sweep/worker%d.log" % (WORK, gpu), errors="ignore").read().splitlines()
        print("[%4.0f min] worker %d: %s" % ((time.time() - t0) / 60, gpu, lines[-1][:150] if lines else "(starting)"), flush=True)
for gpu, p in procs:
    print("worker", gpu, "exit code", p.returncode)
    if p.returncode:
        print(open("%s/sweep/worker%d.log" % (WORK, gpu), errors="ignore").read()[-3000:])

## 3. Score every configuration on the same 50 questions

In [ ]:
import re
import numpy as np, pandas as pd
import eval_harness as H
from rouge_score import rouge_scorer

items = {it["qid"]: it for it in payload["items"]}
ids = list(items)
refs = {q: items[q]["reference"] for q in ids}
gen = {}
for f in glob.glob(WORK + "/sweep/gen_gpu*.jsonl"):
    for l in open(f, encoding="utf-8"):
        if l.strip():
            d = json.loads(l); gen[d["uid"]] = d

WORD = re.compile(r"\S+")
def is_loop(text, n=8, min_rep=3):
    w = [x.lower().strip(".,;:*#-()") for x in WORD.findall(text)]
    c = C.Counter(tuple(w[i:i + n]) for i in range(len(w) - n + 1))
    return bool(c) and max(c.values()) >= min_rep

systems = {"B0a copy nearest answer": {q: items[q]["b0a"] for q in ids},
           "Phase5 saved (greedy rp1.05, cap 768)": {q: items[q]["phase5_text"] for q in ids}}
extra = {"B0a copy nearest answer": {}, "Phase5 saved (greedy rp1.05, cap 768)": {"hit_cap": sum(items[q]["phase5_hit_limit"] for q in ids)}}
for c in ALL:
    got = [gen.get("%s|%s" % (c, q)) for q in ids]
    if not all(got):
        print("WARNING: config %s incomplete (%d/%d) - re-run to resume" % (c, sum(map(bool, got)), len(ids)))
        continue
    systems[c] = {q: (g["text"] if g["text"].strip() else items[q]["b0a"]) for q, g in zip(ids, got)}
    extra[c] = {"hit_cap": sum(g["hit_limit"] for g in got), "sec_per_answer": float(np.mean([g["sec_per_answer"] for g in got])),
                "empty": sum(not g["text"].strip() for g in got)}

res = {n: H.evaluate(p, refs) for n, p in systems.items()}
for n, p in systems.items():
    res[n].update(extra[n])
    res[n]["loops_8gram_x3"] = sum(is_loop(p[q]) for q in ids)
    res[n]["median_words"] = float(np.median([len(p[q].split()) for q in ids]))
cols = ["rougeL", "rouge1", "rouge2", "bleu", "fact_f1", "len_ratio_mean", "median_words", "hit_cap", "loops_8gram_x3", "empty", "sec_per_answer"]
table = pd.DataFrame({n: {c: r.get(c, np.nan) for c in cols} for n, r in res.items()}).T
pd.set_option("display.width", 250)
print("N = %d hold-out questions (reference median %d words)" % (len(ids), np.median([len(refs[q].split()) for q in ids])))
print(table.round(4).to_string())

sc = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)
per = {n: np.array([sc.score(refs[q], p[q])["rougeL"].fmeasure for q in ids]) for n, p in systems.items()}
idx = np.random.default_rng(0).integers(0, len(ids), (2000, len(ids)))
def diff_ci(a, b):
    d = (per[a] - per[b])[idx].mean(axis=1)
    return "%+.4f  [95%% CI %+.4f, %+.4f]" % (per[a].mean() - per[b].mean(), np.percentile(d, 2.5), np.percentile(d, 97.5))
P5N, B0N = "Phase5 saved (greedy rp1.05, cap 768)", "B0a copy nearest answer"
boot = {}
print("\nROUGE-L paired bootstrap (N=%d):" % len(ids))
for n in systems:
    if n not in (P5N, B0N):
        boot[n] = {"minus_phase5": diff_ci(n, P5N), "minus_B0a": diff_ci(n, B0N)}
        print("  %-30s vs Phase5 %s | vs B0a %s" % (n, boot[n]["minus_phase5"], boot[n]["minus_B0a"]))
json.dump({"n": len(ids), "configs": {c: DS.settings(c) for c in ALL}, "results": res, "bootstrap_rougeL": boot},
          open(WORK + "/sweep_results.json", "w"), indent=1, default=float)
print("\nwrote sweep_results.json")

## 4. Look at the question where Phase 5 looped worst, under each setting

In [ ]:
def rep_score(t):
    w = t.lower().split(); c = C.Counter(tuple(w[i:i + 8]) for i in range(len(w) - 7))
    return max(c.values()) if c else 0
q = max(ids, key=lambda q: rep_score(items[q]["phase5_text"]))
print("Q:", items[q]["question"])
for n, p in systems.items():
    t = p[q].replace("\n", " ")
    print("=" * 100, "\n%s (%d words, rougeL %.3f)\n  ...%s" % (n, len(t.split()), per[n][ids.index(q)], t[-500:]))

## After it finishes
Send me the **table from section 3** and the **bootstrap lines** (or just tell me the notebook name; I can download the output).
Output tab: `sweep_results.json`, `sweep/gen_gpu*.jsonl` (every answer).